# V114 - Submission stability audit

Pre-registered plan: v113 adopted on OOF evidence, but the submission rows come from one seed fit. Refit the bagged specialist under five seeds and measure agreement of the 33 test rows: mean pairwise top-1 agreement at least 0.8 and mean rank correlation at least 0.9, fixed up front. A pass confirms the file. A fail withdraws it back to the champion no matter what v113 said, because an unstable weapon misfires on contact.

In [1]:
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

SEEDS = [11, 22, 33, 44, 55]
BAGS = 10
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v114")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Five refits, pairwise agreement, verdict

Specialist trains on all 117 no-chat users per seed. Agreement is measured on ordering, which is all NDCG consumes.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
A = (S - S.mean(0)) / (S.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
is_nochat = np.array([u not in by_user for u in all_users])
nc_tr, nc_te = np.where(is_nochat[:n_train])[0], np.where(is_nochat[n_train:])[0]

runs = []
for s in SEEDS:
    r = np.random.RandomState(s)
    P = np.zeros((len(nc_te), len(MODS)))
    for b in range(BAGS):
        idx = nc_tr[r.choice(len(nc_tr), len(nc_tr), replace=True)]
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(A_tr[idx], Y[idx, j])
            P[:, j] += m.predict(A_te[nc_te]) / BAGS
    runs.append(P)
    print(f"seed {s} done", flush=True)
sp = np.mean([spearmanr(a.ravel(), b.ravel())[0] for a, b in itertools.combinations(runs, 2)])
agree = float(np.mean([(a.argmax(1) == b.argmax(1)).mean() for a, b in itertools.combinations(runs, 2)]))
print(f"top1-agreement={agree:.3f} rank-corr={sp:.3f}")
print("SUBMISSION CONFIRMED" if (agree >= 0.8 and sp >= 0.9) else "SUBMISSION WITHDRAWN: rows unstable across seeds")

seed 11 done


seed 22 done


seed 33 done


seed 44 done


seed 55 done


top1-agreement=0.752 rank-corr=0.967
SUBMISSION WITHDRAWN: rows unstable across seeds
